# IntelliDoc AI — Notebook 5: LayoutLMv3 Pretrained Baseline on CORD

**Purpose:** Run pretrained `microsoft/layoutlmv3-base` on the **CORD (Consolidated Receipt Dataset)** test set **without any receipt fine-tuning** and record baseline accuracy scores.

**Research Objective:**
1. Measures zero-shot / baseline transfer performance on commercial retail receipts.
2. Establishes the empirical baseline before fine-tuning on receipt line-items (Notebook 6).

---
> **Prerequisites:** Ensure CORD is uploaded to Google Drive (`/content/drive/MyDrive/Intellidoc_AI/datasets_public/cord`).
> **Runtime:** ~5–10 minutes on a free Google Colab T4 GPU.

## Step 1: Mount Google Drive, Set Paths, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Check path
CORD_ROOT = '/content/drive/MyDrive/Intellidoc_AI/datasets_public/cord'
if not os.path.exists(CORD_ROOT):
    # Fallback to root cord folder
    CORD_ROOT = '/content/drive/MyDrive/cord'

TEST_DIR = os.path.join(CORD_ROOT, 'test')
if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(f'Missing CORD test split: {TEST_DIR}')

print(f'✅ Paths verified. CORD Test Directory: {TEST_DIR}')

# Install transformers, torch, datasets, seqeval
!pip install -q transformers datasets seqeval Pillow
print('✅ Required libraries installed!')

## Step 2: CORD Label Scheme (BIO Tagging)
Map CORD receipt entities (`menu`, `sub_total`, `total`, `void_total`) to BIO tags.

In [ ]:
# Standard top-level receipt categories in CORD
ENTITIES = ['MENU', 'SUBTOTAL', 'TOTAL', 'VOID']

# BIO Tagging labels
LABELS = ['O']
for ent in ENTITIES:
    LABELS.append(f'B-{ent}')
    LABELS.append(f'I-{ent}')

ID2LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f'Total Classification Classes: {NUM_LABELS}')
print('Labels:', LABELS)

## Step 3: Load & Preprocess CORD Test Split
Load test receipt images and quadrilaterals, normalizing bounding coordinates to $[0, 1000]$.

In [ ]:
import json
import glob
from PIL import Image

def normalize_bbox(quad, width, height):
    # quad has x1, y1, x2, y2, x3, y3, x4, y4
    xs = [quad.get('x1', 0), quad.get('x2', 0), quad.get('x3', 0), quad.get('x4', 0)]
    ys = [quad.get('y1', 0), quad.get('y2', 0), quad.get('y3', 0), quad.get('y4', 0)]
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(ys), max(ys)
    return [
        int(1000 * (x0 / width)),
        int(1000 * (y0 / height)),
        int(1000 * (x1 / width)),
        int(1000 * (y1 / height))
    ]

def map_category_to_entity(cat):
    if 'menu' in cat:
        return 'MENU'
    elif 'sub_total' in cat:
        return 'SUBTOTAL'
    elif 'total' in cat:
        return 'TOTAL'
    elif 'void' in cat:
        return 'VOID'
    return None

def load_cord_samples(split_dir, limit=100):
    json_files = sorted(glob.glob(os.path.join(split_dir, 'json', '*.json')))[:limit]
    samples = []
    for j_path in json_files:
        stem = os.path.basename(j_path).replace('.json', '')
        img_path = os.path.join(split_dir, 'image', stem + '.png')
        if not os.path.exists(img_path):
            img_path = os.path.join(split_dir, 'image', stem + '.jpg')
        if not os.path.exists(img_path):
            continue
        
        with open(j_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        img = Image.open(img_path).convert('RGB')
        width, height = img.size
        words, bboxes, labels = [], [], []
        
        for line in data.get('valid_line', []):
            cat = line.get('category', '')
            entity = map_category_to_entity(cat)
            line_words = line.get('words', [])
            for i, w in enumerate(line_words):
                t = w.get('text', '').strip()
                if not t:
                    continue
                quad = w.get('quad', {})
                if not quad:
                    continue
                box = normalize_bbox(quad, width, height)
                # Clamp
                box = [max(0, min(1000, b)) for b in box]
                
                # Tag prefix
                tag = 'O'
                if entity:
                    prefix = 'B' if i == 0 else 'I'
                    tag = f'{prefix}-{entity}'
                
                words.append(t)
                bboxes.append(box)
                labels.append(LABEL2ID.get(tag, 0))
        
        if words:
            samples.append({
                'image': img,
                'words': words,
                'bboxes': bboxes,
                'labels': labels
            })
    return samples

test_samples = load_cord_samples(TEST_DIR)
print(f'✅ Loaded {len(test_samples)} receipt samples from test split.')

## Step 4: Run Pretrained LayoutLMv3 Baseline Inference
Evaluate out-of-the-box `microsoft/layoutlmv3-base` on CORD test receipts.

In [ ]:
import torch
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')

MODEL_NAME = 'microsoft/layoutlmv3-base'
processor = LayoutLMv3Processor.from_pretrained(MODEL_NAME, apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
).to(device)
model.eval()

all_preds, all_labels = [], []

print('Evaluating pretrained baseline on CORD receipts...')
with torch.no_grad():
    for sample in tqdm(test_samples):
        encoding = processor(
            sample['image'],
            sample['words'],
            boxes=sample['bboxes'],
            word_labels=sample['labels'],
            truncation=True,
            padding='max_length',
            max_length=512,
            return_tensors='pt'
        )
        
        input_ids = encoding['input_ids'].to(device)
        bbox = encoding['bbox'].to(device)
        pixel_values = encoding['pixel_values'].to(device)
        target_labels = encoding['labels'].squeeze(0).tolist()
        
        outputs = model(input_ids=input_ids, bbox=bbox, pixel_values=pixel_values)
        preds = torch.argmax(outputs.logits, dim=-1).squeeze(0).tolist()
        
        true_seq, pred_seq = [], []
        for p, t in zip(preds, target_labels):
            if t != -100:  # Ignore padded / subword tokens
                true_seq.append(ID2LABEL[t])
                pred_seq.append(ID2LABEL[p])
        
        if true_seq:
            all_labels.append(true_seq)
            all_preds.append(pred_seq)

print('\n' + '=' * 60)
print('=== CORD LayoutLMv3 Pretrained Baseline Evaluation Results ===')
print('=' * 60)
print(classification_report(all_labels, all_preds, zero_division=0))
print(f'Overall Baseline F1 Score: {f1_score(all_labels, all_preds, zero_division=0):.4f}')